# 02 — Twitter/X Dataset Inspection and Cleaning

**Project:** From Online Attention to Electoral Outcomes: A Network Science Analysis of Philippine Election 2025 Twitter/X Communication

This notebook is part of the notebook set for the Philippine Election 2025 Twitter/X network science project.



## Visible progress tracker

This notebook now prints numbered stage updates while it runs. If Jupyter shows `[*]`, the current stage is still processing. A run log is also saved under `outputs/run_logs/`.


In [ ]:
# VISIBLE PROGRESS TRACKER — automatically added in v5
from pathlib import Path as _ProgressPath
import sys as _progress_sys
_PROGRESS_ROOT = _ProgressPath.cwd().parent if _ProgressPath.cwd().name == "notebooks" else _ProgressPath.cwd()
_progress_sys.path.append(str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("02_twitter_dataset_inspection_cleaning", total_steps=9, root=_PROGRESS_ROOT)
except Exception as _progress_error:
    print(f"Progress tracker could not start: {_progress_error}")
    class _FallbackProgress:
        def __init__(self): self.current = 0
        def step(self, label):
            self.current += 1
            print(f"[stage {self.current}] {label}", flush=True)
        def info(self, label): print(f"[info] {label}", flush=True)
        def done(self, label="Notebook completed"): print(f"✓ {label}", flush=True)
    progress = _FallbackProgress()


## Purpose

This notebook cleans the Twitter/X dataset, parses time fields, audits missingness/duplicates, and creates the base dataset used by later notebooks.

In [ ]:
progress.step("Purpose")
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

# Make ../src importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))

from election_network_utils import *
paths = ensure_dirs(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"

RAW = paths["raw"]
PROCESSED = paths["processed"]
FIGURES = paths["figures"]
INTERACTIVE = paths["interactive"]
TABLES = paths["tables"]
NETWORKS = paths["networks"]
REPORT_ASSETS = paths["report_assets"]

print("Project root:", ROOT)


In [ ]:
progress.step("Purpose")
tweets_path = RAW / "for_export_philippine_elections.csv"
tweets = pd.read_csv(tweets_path)
print("Twitter/X dataset shape:", tweets.shape)
display(tweets.head())


In [ ]:
progress.step("Purpose")
# Parse dates and numeric columns
clean = tweets.copy()
clean["createdAt"] = pd.to_datetime(clean["createdAt"], errors="coerce", utc=True)
clean["date"] = clean["createdAt"].dt.date
clean["hour"] = clean["createdAt"].dt.hour
clean["month"] = clean["createdAt"].dt.to_period("M").astype(str)

engagement_cols = ["retweetCount", "replyCount", "likeCount", "quoteCount", "viewCount", "bookmarkCount"]
for c in engagement_cols:
    clean[c] = pd.to_numeric(clean[c], errors="coerce").fillna(0)
clean["total_engagement"] = clean[["retweetCount", "replyCount", "likeCount", "quoteCount", "bookmarkCount"]].sum(axis=1)
clean["text_length"] = clean["text"].astype(str).str.len()
clean["text_clean"] = clean["text"].apply(normalize_text)
clean["hashtags"] = clean["text"].apply(extract_hashtags)
clean["mentions"] = clean["text"].apply(extract_mentions)
clean["domains"] = clean["text"].apply(extract_domains)
clean["hashtag_count"] = clean["hashtags"].apply(len)
clean["mention_count"] = clean["mentions"].apply(len)
clean["domain_count"] = clean["domains"].apply(len)

print("Date range:", clean["createdAt"].min(), "to", clean["createdAt"].max())
print("Unique authors:", clean["pseudo_author_userName"].nunique())
clean.head()


In [ ]:
progress.step("Purpose")
# Data quality audit
missing = clean.isna().mean().reset_index()
missing.columns = ["column", "missing_rate"]
missing = missing.sort_values("missing_rate", ascending=False)
missing.to_csv(TABLES / "02_missing_value_rates.csv", index=False)

duplicates = pd.DataFrame([
    {"metric": "rows", "value": len(clean)},
    {"metric": "duplicate_pseudo_id", "value": clean["pseudo_id"].duplicated().sum() if "pseudo_id" in clean.columns else np.nan},
    {"metric": "duplicate_text", "value": clean["text"].duplicated().sum() if "text" in clean.columns else np.nan},
    {"metric": "missing_createdAt", "value": clean["createdAt"].isna().sum()},
    {"metric": "missing_text", "value": clean["text"].isna().sum()},
])
duplicates.to_csv(TABLES / "02_duplicate_and_missing_summary.csv", index=False)
display(missing.head(15))
display(duplicates)


In [ ]:
progress.step("Purpose")
# Save cleaned base file
clean.to_csv(PROCESSED / "tweets_cleaned_base.csv", index=False)
print("Saved:", PROCESSED / "tweets_cleaned_base.csv")


In [ ]:
progress.step("volume = clean.groupby('date', as_index=False).size().rename(columns={'size': 'tweet_count'})")
# Tweet volume timeline
volume = clean.groupby("date", as_index=False).size().rename(columns={"size": "tweet_count"})
volume.to_csv(PROCESSED / "tweet_volume_by_date.csv", index=False)
fig = px.line(volume, x="date", y="tweet_count", markers=True, title="Tweet volume over time")
fig.update_layout(height=520, xaxis_title="Date", yaxis_title="Tweet count")
save_plotly(fig, INTERACTIVE / "02_tweet_volume_timeline.html", FIGURES / "02_tweet_volume_timeline.png")
fig.show()


In [ ]:
progress.step("hourly = clean.groupby(['date', 'hour'], as_index=False).size().rename(columns={'size': 'tweet_count'})")
# Activity by day and hour
hourly = clean.groupby(["date", "hour"], as_index=False).size().rename(columns={"size": "tweet_count"})
fig = px.density_heatmap(hourly, x="hour", y="date", z="tweet_count", title="Tweet activity heatmap by date and hour")
fig.update_layout(height=720, xaxis_title="Hour UTC", yaxis_title="Date")
save_plotly(fig, INTERACTIVE / "02_tweet_activity_heatmap.html", FIGURES / "02_tweet_activity_heatmap.png")
fig.show()


In [ ]:
progress.step("lang_counts = clean['lang'].fillna('unknown').value_counts().reset_index()")
# Language distribution
lang_counts = clean["lang"].fillna("unknown").value_counts().reset_index()
lang_counts.columns = ["language", "tweet_count"]
lang_counts.to_csv(TABLES / "02_language_distribution.csv", index=False)
fig = px.bar(lang_counts.head(15), x="language", y="tweet_count", title="Top language labels in the dataset")
fig.update_layout(height=480)
save_plotly(fig, INTERACTIVE / "02_language_distribution.html", FIGURES / "02_language_distribution.png")
fig.show()


In [ ]:
progress.step("eng_long = clean[engagement_cols + ['total_engagement']].melt(var_name='metric', value_name='value')")
# Engagement distribution using log transform for readability
eng_long = clean[engagement_cols + ["total_engagement"]].melt(var_name="metric", value_name="value")
eng_long["log1p_value"] = np.log1p(eng_long["value"])
plt.figure(figsize=(12, 6))
sns.boxplot(data=eng_long, x="metric", y="log1p_value")
plt.title("Distribution of engagement metrics, log1p scale")
plt.xlabel("Metric")
plt.ylabel("log(1 + value)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(FIGURES / "02_engagement_distribution_log_boxplot.png", dpi=220, bbox_inches="tight")
plt.show()


## Run complete

The final cell closes the progress bar and confirms that all tracked notebook stages finished.


In [ ]:
progress.done("Notebook run completed")
